# NASA PROMISE CM1 & KC1 Defect Dataset Inspection (P3-01)

## Overview
This notebook inspects the **CM1** and **KC1** software defect datasets from the **NASA Metrics Data Program (MDP)**, hosted by the **PROMISE Software Engineering Repository**.

### Dataset Metadata & Licensing
- **CM1**: NASA spacecraft instrument system written in C (498 modules/functions).
- **KC1**: NASA ground system for receiving and processing satellite data written in C++ (2,109 modules/methods).
- **License**: Public Domain (NASA Open Science / PROMISE Repository, CC0-compatible for empirical research).
- **Features**: 21 static code metrics (McCabe cyclomatic/essential/design complexity, Halstead software science suite, lines of code).
- **Target**: `defects` (boolean binary classification: 0 = non-defective, 1 = defective).

In [ ]:
import pandas as pd

from packages.ml.dataset import load_cm1, load_kc1
from packages.ml.inspector import inspect_dataset

cm1 = load_cm1()
kc1 = load_kc1()

print(f"CM1 Shape: {cm1.shape}")
print(f"KC1 Shape: {kc1.shape}")

## 1. Class Distribution & Imbalance Analysis

Software defect datasets exhibit severe class imbalance, since the vast majority of software modules are defect-free.

In [ ]:
cm1_report = inspect_dataset(cm1, "cm1")
kc1_report = inspect_dataset(kc1, "kc1")

imbalance_df = pd.DataFrame(
    [
        {
            "Dataset": "CM1",
            "Total Rows": cm1_report.rows,
            "Clean (0)": cm1_report.negative_count,
            "Defective (1)": cm1_report.positive_count,
            "Defect Rate (%)": f"{cm1_report.positive_percentage:.2f}%",
            "Imbalance Ratio": f"{cm1_report.imbalance_ratio:.2f}:1",
        },
        {
            "Dataset": "KC1",
            "Total Rows": kc1_report.rows,
            "Clean (0)": kc1_report.negative_count,
            "Defective (1)": kc1_report.positive_count,
            "Defect Rate (%)": f"{kc1_report.positive_percentage:.2f}%",
            "Imbalance Ratio": f"{kc1_report.imbalance_ratio:.2f}:1",
        },
    ]
)
imbalance_df

## 2. Duplicate & Inconsistency Analysis

Empirical software engineering literature (e.g. Shepperd et al., 2013) identified significant duplication and contradictory instances in the raw NASA MDP datasets.
- **Exact duplicates**: Identical feature vector and identical defect label.
- **Contradictory instances**: Identical feature vector but differing defect labels (labelling noise).

In [ ]:
dup_df = pd.DataFrame(
    [
        {
            "Dataset": "CM1",
            "Exact Duplicates": cm1_report.exact_duplicates_count,
            "Exact Dup %": f"{cm1_report.exact_duplicates_pct:.2f}%",
            "Feature Duplicates": cm1_report.feature_duplicates_count,
            "Contradictory Rows": cm1_report.contradictory_instances_count,
        },
        {
            "Dataset": "KC1",
            "Exact Duplicates": kc1_report.exact_duplicates_count,
            "Exact Dup %": f"{kc1_report.exact_duplicates_pct:.2f}%",
            "Feature Duplicates": kc1_report.feature_duplicates_count,
            "Contradictory Rows": kc1_report.contradictory_instances_count,
        },
    ]
)
dup_df

## 3. Missing Value Audit
Check for NaN, null, or empty fields across all 21 metric features and target.

In [ ]:
print("CM1 Missing values total:", cm1.isnull().sum().sum())
print("KC1 Missing values total:", kc1.isnull().sum().sum())

## 4. Key Takeaways for Modeling (Phase 3 P3-02+)
1. **Never use raw accuracy**: A trivial dummy classifier predicting all clean achieves ~90% accuracy on CM1 and ~85% on KC1.
2. **Primary metrics**: Use **ROC-AUC**, **PR-AUC**, Precision, Recall, and balanced F1 score.
3. **Validation strategy**: Use `StratifiedKFold` (5 folds) with leakage-safe `Pipeline`.
4. **Feature preprocessing**: Scale features with `RobustScaler` or `StandardScaler` to handle extreme right-skewed Halstead metrics.